# BERT Intent Classification

## Objective
Train a transformer model for customer intent classification and compare it with Logistic Regression and Linear SVM.

Model:
- DistilBERT

Task:
- Multi-class Intent Classification

Classes:
- 27

In [21]:
import pandas as pd
import numpy as np

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [22]:
df = pd.read_csv(
    "../data/raw/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
)

df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [23]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df["intent"])

num_labels = len(label_encoder.classes_)
print(num_labels)

27


In [24]:
train_df,test_df=train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['label']
)
print(train_df.shape)
print(test_df.shape)

(21497, 6)
(5375, 6)


In [25]:
train_dataset = Dataset.from_pandas(train_df)

test_dataset = Dataset.from_pandas(test_df)

In [26]:
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(
    checkpoint
)

In [27]:
def tokenize(batch):

    return tokenizer(

        batch["instruction"],

        padding="max_length",

        truncation=True,

        max_length=64

    )

In [28]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/21497 [00:00<?, ? examples/s]

Map:   0%|          | 0/5375 [00:00<?, ? examples/s]

In [30]:
train_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

test_dataset.set_format(
    "torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)

In [31]:
model = AutoModelForSequenceClassification.from_pretrained(

    checkpoint,

    num_labels=num_labels
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [34]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../reports/bert_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,

    logging_steps=100,

    save_total_limit=2,

    report_to="none"
)

In [35]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

In [36]:
def compute_metrics(eval_pred):
    logits,labels =eval_pred

    predictions = np.argmax(logits,axis=1)

    return accuracy.compute(
        predictions = predictions,
        references=labels
    )

In [37]:
from transformers import Trainer

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    compute_metrics=compute_metrics,

)

In [38]:
trainer.train()

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.016861,0.019234,0.996279
2,0.006487,0.012299,0.997395
3,0.001345,0.010568,0.997767


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4032, training_loss=0.17320220969757805, metrics={'train_runtime': 6100.3832, 'train_samples_per_second': 10.572, 'train_steps_per_second': 0.661, 'total_flos': 1068345474198912.0, 'train_loss': 0.17320220969757805, 'epoch': 3.0})

In [39]:
trainer.evaluate()

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.001345,0.010568,3,0.997767


{'eval_loss': 0.010568398982286453, 'eval_accuracy': 0.9977674418604651}

In [40]:
model.save_pretrained("../backend/app/ai/bert_model")

tokenizer.save_pretrained("../backend/app/ai/bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('../backend/app/ai/bert_model\\tokenizer_config.json',
 '../backend/app/ai/bert_model\\tokenizer.json')